In [ ]:
pip install pybaseball

In [ ]:
import pandas as pd
import unicodedata

def normalize_name(name):
    """Removes accents and extra spaces (e.g., 'Rodríguez' -> 'Rodriguez')."""
    if not isinstance(name, str): return ""
    name = unicodedata.normalize('NFD', name).encode('ascii', 'ignore').decode("utf-8")
    return name.strip()

# 1. Load your actual file
# Note: Ensure the file name matches exactly what is in your folder
df = pd.read_csv('projections.csv')

# 2. Expanded Top Position Map (Top 75ish most important targets)
top_positions = {
    'Aaron Judge': 'OF', 'Bobby Witt Jr.': 'SS', 'Gunnar Henderson': 'SS',
    'Juan Soto': 'OF', 'Jose Ramirez': '3B', 'Shohei Ohtani': 'Util',
    'Yordan Alvarez': 'OF', 'Kyle Tucker': 'OF', 'Vladimir Guerrero Jr.': '1B',
    'Trea Turner': 'SS', 'Francisco Lindor': 'SS', 'Rafael Devers': '3B',
    'Freddie Freeman': '1B', 'Corey Seager': 'SS', 'Austin Riley': '3B',
    'Matt Olson': '1B', 'Ozzie Albies': '2B', 'Marcus Semien': '2B',
    'Pete Alonso': '1B', 'Adley Rutschman': 'C', 'Jose Altuve': '2B',
    'Julio Rodriguez': 'OF', 'Ronald Acuna Jr.': 'OF', 'Fernando Tatis Jr.': 'OF',
    'Riley Greene': 'OF', 'Elly De La Cruz': 'SS', 'Bryce Harper': '1B',
    'Mookie Betts': 'OF', 'William Contreras': 'C', 'Ketel Marte': '2B',
    'Manny Machado': '3B', 'Corbin Carroll': 'OF', 'Alex Bregman': '3B',
    'Jose Ramirez': '3B', 'Cal Raleigh': 'C', 'Jackson Chourio': 'OF'
}

# Normalize the keys in our dictionary for better matching
top_positions = {normalize_name(k): v for k, v in top_positions.items()}

def assign_pos(row):
    # Normalize the name from the CSV
    raw_name = normalize_name(row['Name'])

    # Check our dictionary
    if raw_name in top_positions:
        return top_positions[raw_name]

    # Casing-safe check for Pitchers
    if str(row['POS']).strip() == 'Pitcher':
        return 'P'

    # Check for Catchers (often labeled 'C' in some projections)
    if 'C' in str(row['POS']):
        return 'C'

    # Fallback
    return 'Util'

# 3. Apply the logic to a NEW column
df['Draft_POS'] = df.apply(assign_pos, axis=1)

# 4. Save to a new file
df.to_csv('projections_with_pos.csv', index=False)

# Verification
print("--- Check: First 10 Rows of new column ---")
print(df[['Name', 'Draft_POS']].head(10))

--- Check: First 10 Rows of new column ---
                 Name Draft_POS
0         Aaron Judge        OF
1      Bobby Witt Jr.        SS
2    Gunnar Henderson        SS
3         Cal Raleigh         C
4     Julio Rodríguez        OF
5           Juan Soto        OF
6       Shohei Ohtani      Util
7    Ronald Acuña Jr.        OF
8  Fernando Tatis Jr.        OF
9    Francisco Lindor        SS


In [ ]:
import pandas as pd
import unicodedata

class DraftOracle:
    def __init__(self, league_size=10):
        self.league_size = league_size
        # Point weights based on your 10-team points league
        self.hitter_weights = {'R': 1.9, '1B': 2.6, '2B': 5.2, '3B': 7.8, 'HR': 10.4, 'RBI': 1.9, 'SB': 4.2, 'BB': 2.6, 'HBP': 2.6}
        self.pitcher_weights = {'W': 8, 'SV': 8, 'OUT': 1, 'H': -1.3, 'ER': -3, 'BB': -1.3, 'HBP': -1.3, 'K': 3}

    def calculate_points(self, row):
        if row['POS'] == 'P':
            return sum(row.get(s, 0) * w for s, w in self.pitcher_weights.items())
        return sum(row.get(s, 0) * w for s, w in self.hitter_weights.items())

    def get_recommendations(self, df, team_needs):
        vorp_list = []
        for pos in team_needs:
            pos_players = df[df['POS'] == pos].sort_values(by='Total_Points', ascending=False)
            if not pos_players.empty:
                # VORP Baseline: The 10th best player at that position
                rep_idx = min(self.league_size - 1, len(pos_players) - 1)
                rep_val = pos_players.iloc[rep_idx]['Total_Points']

                top_p = pos_players.iloc[0].copy()
                top_p['VORP'] = round(top_p['Total_Points'] - rep_val, 2)

                # Tier Drop Logic: How much do you lose if you don't pick THIS player now?
                if len(pos_players) > 1:
                    top_p['Tier_Drop'] = round(top_p['Total_Points'] - pos_players.iloc[1]['Total_Points'], 2)
                else:
                    top_p['Tier_Drop'] = 0

                vorp_list.append(top_p)
        return pd.DataFrame(vorp_list).sort_values(by='VORP', ascending=False)

def normalize(text):
    return unicodedata.normalize('NFD', str(text)).encode('ascii', 'ignore').decode("utf-8").lower().strip()

# --- SETUP ---
df = pd.read_csv('projections_with_pos.csv')
df = df[~((df['Name'].str.contains('Ohtani')) & (df['Draft_POS'] != 'P'))]
if 'Draft_POS' in df.columns:
    df['POS'] = df['Draft_POS']

oracle = DraftOracle(league_size=10)
df['Total_Points'] = df.apply(oracle.calculate_points, axis=1)
my_needs = ['1B', '2B', '3B', 'SS', 'OF', 'Util', 'P']

print("\n--- DIAMONDLOGIC VORP CONSOLE LOADED ---")

while True:
    recs = oracle.get_recommendations(df, my_needs)
    print("\n" + "="*50)
    print(f"{'NAME':<20} | {'POS':<5} | {'POINTS':<8} | {'VORP':<8} | {'DROP'}")
    print("-" * 50)
    for _, row in recs.iterrows():
        print(f"{row['Name']:<20} | {row['POS']:<5} | {row['Total_Points']:<8.1f} | {row['VORP']:<8.1f} | {row['Tier_Drop']}")

    val = input("\nEnter name pick (or 'exit'): ").strip()
    if val.lower() == 'exit': break

    search = normalize(val)
    df['search_name'] = df['Name'].apply(normalize)
    matches = df[df['search_name'].str.contains(search)]

    if len(matches) == 1:
        p_name = matches.iloc[0]['Name']
        df = df[df['Name'] != p_name]
        print(f"✅ Removed: {p_name}")
    elif len(matches) > 1:
        print(f"⚠️ Multiple matches: {matches['Name'].tolist()}")
    else:
        print(f"❌ '{val}' not found.")


--- DIAMONDLOGIC VORP CONSOLE LOADED ---

NAME                 | POS   | POINTS   | VORP     | DROP
--------------------------------------------------
Aaron Judge          | OF    | 875.5    | 303.4    | 69.3
Bobby Witt Jr.       | SS    | 778.6    | 226.7    | 35.1
José Ramírez         | 3B    | 731.7    | 174.0    | 106.3
Kyle Schwarber       | Util  | 786.1    | 146.2    | 24.1
Ketel Marte          | 2B    | 609.1    | 145.2    | 71.4
Pete Alonso          | 1B    | 733.8    | 107.1    | 13.3
Edwin Díaz           | P     | 320.0    | 64.0     | 24.0

Enter name pick (or 'exit'): judge 
✅ Removed: Aaron Judge

NAME                 | POS   | POINTS   | VORP     | DROP
--------------------------------------------------
Juan Soto            | OF    | 806.2    | 239.1    | 16.3
Bobby Witt Jr.       | SS    | 778.6    | 226.7    | 35.1
José Ramírez         | 3B    | 731.7    | 174.0    | 106.3
Kyle Schwarber       | Util  | 786.1    | 146.2    | 24.1
Ketel Marte          | 2B    | 609.1  